In [1]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime

# Load all JSON files
output_dir = Path('.')
results = {}

for json_file in sorted(output_dir.glob('*.json')):
    with open(json_file) as f:
        data = json.load(f)
        
        # Use filename as ID
        experiment_id = json_file.stem
        
        results[experiment_id] = data

print(f"Loaded {len(results)} experiments:")
for exp_id in results.keys():
    print(f"  - {exp_id}")

Loaded 3 experiments:
  - baseline_30_azure-openai-foundry_gpt-4_1
  - full_30_azure-openai-foundry_gpt-4_1
  - guidelines_30_azure-openai-foundry_gpt-4_1


In [ ]:
# Create comparison tables
datasets = list(next(iter(results.values()))['datasets'].keys())

# Table 1: Macro F1 scores across datasets
f1_data = []
for model_name, model_data in results.items():
    row = {'Model': model_name}
    for dataset in datasets:
        if dataset in model_data['datasets']:
            f1 = model_data['datasets'][dataset]['reports']['original']['macro avg']['f1-score']
            row[dataset] = round(f1, 3)
    f1_data.append(row)

df_f1 = pd.DataFrame(f1_data).set_index('Model')
print("\n=== Macro F1 Scores by Model and Dataset ===")
print(df_f1.to_string())

# Table 2: Accuracy across datasets
acc_data = []
for model_name, model_data in results.items():
    row = {'Model': model_name}
    for dataset in datasets:
        if dataset in model_data['datasets']:
            acc = model_data['datasets'][dataset]['reports']['original']['accuracy']
            row[dataset] = round(acc, 3)
    acc_data.append(row)

df_acc = pd.DataFrame(acc_data).set_index('Model')
print("\n=== Accuracy by Model and Dataset ===")
print(df_acc.to_string())

# Summary statistics
print("\n=== Average Performance Across All Datasets ===")
summary = pd.DataFrame({
    'Model': list(results.keys()),
    'Avg Macro F1': [df_f1.loc[model].mean() for model in results.keys()],
    'Avg Accuracy': [df_acc.loc[model].mean() for model in results.keys()]
}).set_index('Model')
print(summary.round(3).to_string())

# Optional: Display styled dataframes if running in Jupyter
display(df_f1.style.highlight_max(axis=0, color='lightgreen').format("{:.3f}"))


=== Macro F1 Scores by Model and Dataset ===
                                            ABSTRCT  ACQUA    AEC    AFS  ARGUMINSCI  FINARG    IAM     PE  SCIARK  USELEC
Model                                                                                                                     
baseline_30_azure-openai-foundry_gpt-4_1      0.729  0.766  0.402  0.653       0.764   0.499  0.667  0.433   0.593   0.722
full_30_azure-openai-foundry_gpt-4_1          0.729    NaN    NaN    NaN       0.733     NaN    NaN  0.542     NaN   0.697
guidelines_30_azure-openai-foundry_gpt-4_1    0.729    NaN    NaN    NaN       0.764     NaN    NaN  0.486     NaN   0.732

=== Accuracy by Model and Dataset ===
                                            ABSTRCT  ACQUA  AEC    AFS  ARGUMINSCI  FINARG    IAM     PE  SCIARK  USELEC
Model                                                                                                                   
baseline_30_azure-openai-foundry_gpt-4_1      0.733  0.767

,ABSTRCT,ACQUA,AEC,AFS,ARGUMINSCI,FINARG,IAM,PE,SCIARK,USELEC
Model,,,,,,,,,,
baseline_30_azure-openai-foundry_gpt-4_1,0.729,0.766,0.402,0.653,0.764,0.499,0.667,0.433,0.593,0.722
full_30_azure-openai-foundry_gpt-4_1,0.729,nan,nan,nan,0.733,nan,nan,0.542,nan,0.697
guidelines_30_azure-openai-foundry_gpt-4_1,0.729,nan,nan,nan,0.764,nan,nan,0.486,nan,0.732


,ABSTRCT,ACQUA,AEC,AFS,ARGUMINSCI,FINARG,IAM,PE,SCIARK,USELEC
Model,,,,,,,,,,
baseline_30_azure-openai-foundry_gpt-4_1,0.733,0.767,0.600,0.667,0.767,0.567,0.667,0.433,0.600,0.733
full_30_azure-openai-foundry_gpt-4_1,0.733,nan,nan,nan,0.733,nan,nan,0.567,nan,0.700
guidelines_30_azure-openai-foundry_gpt-4_1,0.733,nan,nan,nan,0.767,nan,nan,0.500,nan,0.733


In [ ]:
# Robustness Analysis: Macro F1 Deltas (Manipulation - Original)
# Negative delta = performance drop under manipulation (better robustness = less drop)
# Positive delta = performance improvement
# Close to 0 = most robust

manipulation_types = ['feger', 'shuffle']

for manip_type in manipulation_types:
    print(f"\n{'='*60}")
    print(f"=== Robustness to '{manip_type.upper()}' Manipulation ===")
    print(f"{'='*60}")
    
    delta_data = []
    for model_name, model_data in results.items():
        row = {'Model': model_name}
        for dataset in datasets:
            if dataset in model_data['datasets']:
                reports = model_data['datasets'][dataset]['reports']
                if manip_type in reports:
                    original_f1 = reports['original']['macro avg']['f1-score']
                    manip_f1 = reports[manip_type]['macro avg']['f1-score']
                    delta = manip_f1 - original_f1  # Negative = performance drop
                    row[dataset] = round(delta, 3)
        delta_data.append(row)
    
    df_delta = pd.DataFrame(delta_data).set_index('Model')
    print(f"\nMacro F1 Delta ({manip_type.capitalize()} - Original):")
    print(df_delta.to_string())
    
    # Summary: Average delta across all datasets
    print(f"\n=== Average Robustness (Closer to 0 = More Robust) ===")
    avg_delta = pd.DataFrame({
        'Model': list(results.keys()),
        'Avg F1 Delta': [df_delta.loc[model].mean() for model in results.keys()]
    }).set_index('Model').sort_values('Avg F1 Delta', ascending=False)
    print(avg_delta.round(3).to_string())
    
    # Display styled dataframe - reversed colormap so negative (better) is green
    styled = df_delta.style.background_gradient(cmap='RdYlGn_r', axis=None, vmin=-0.5, vmax=0.5)
    display(styled.format("{:.3f}"))